# Classification of Wikipedia Articles - Rule-Based Approaches
Wikipedia is an encyclopedia that covers a large amount of diverse topics. All articles are created, corrected and updated by individuals. The goal is to correctly document as many topics as possible by collecting the knowledge of a large number of people. However, some articles stand out due to their completeness, scope and presentation, and for this they are marked with the distinction of the Excellent Article.

This notebook contains multiple rule-based approaches that aim to classifiy articles wether they are excellent or not. The notebook is structured as follows:

1. [Imports](#1-imports)
2. [Load Preprocessed Data](#2-load-preprocessed-data)
3. [Split Dataset](#3-split-dataset)
4. [Tokenize Dataset](#4-tokenize-dataset)
5. [Naive Bayes Classification](#5-naive-bayes-classification) <br>
    5.1 [Define Naive Bayes Classifier & Parameter Grid](#51-define-naive-bayes-classifier--parameter-grid) <br>
    5.2 [Grid-Search & Cross-Validation](#52-grid-search--cross-validation) <br>
    5.3 [Validate Classification Results](#53-validate-classification-results)
6. [Support Vector Machine Classification](#6-support-vector-machine-classification) <br>
    6.1 [Define SVM Classifier & Parameter Grid](#61-define-svm-classifier--parameter-grid) <br>
    6.2 [Grid-Search & Cross-Validation](#62-grid-search--cross-validation) <br>
    6.3 [Validate Classification Results](#63-validate-classification-results)
7. [Decision Tree Classification](#7-decision-tree-classification) <br>
    7.1 [Define Decision Tree Classifier & Parameter Grid](#71-define-decision-tree-classifier--parameter-grid) <br>
    7.2 [Grid-Search & Cross-Validation](#72-grid-search--cross-validation) <br>
    7.3 [Validate Classification Results](#73-validate-classification-results)
8. [Compare Models](#8-compare-results)
9. [Conclusion](#9-conclusion)

## 1. Imports
If some libraries are not installed, you can use the `requierements.txt` and run
```
$ pip install -r requirements.txt
```
in the terminal at the root of the project. <br>
In addition, a function for the calculation of the classification metrics is defined.

In [11]:
import pandas as pd
import numpy as np

# Import Pre-Processing libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Import classification libraries
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import joblib

# Import hyperparameter-optimization & cross validation libraries
from sklearn.model_selection import GridSearchCV

# Import classification metrics
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, accuracy_score
from imblearn.metrics import geometric_mean_score

# Import visualization libraries
from prettytable import PrettyTable


In [2]:
def calculate_metrics(y_true:list, y_predict:list) -> list:
    f1score = f1_score(y_true, y_predict)
    gm = geometric_mean_score(y_true, y_predict, average="binary")
    auc = roc_auc_score(y_true, y_predict, average="weighted")
    precision = precision_score(y_true, y_predict)
    recall = recall_score(y_true, y_predict)
    accuracy = accuracy_score(y_true, y_predict)
    return [f1score, gm, auc, precision, recall, accuracy]

## 2. Load Preprocessed Data
The next step is to load the preprocessed data, that was created with the `00_Article Preprocessing.ipynb` notebook.

In [3]:
dataframe = pd.read_pickle("../../Data/processed_dataset.pkl")

In [4]:
X = np.array(dataframe["text"].values)
y = np.asanyarray(dataframe["label"].values).astype(np.int16)

In [5]:
np.unique(y, return_counts=True)

(array([0, 1], dtype=int16), array([4193, 2794]))

## 3. Split Dataset
In order to correctly validate the training of the models an enable a validation of the results, the dataset is split into a training-, test- and validation-subset. To avoid a further skew in the distribution of the classes, the `stratify`-option is set to enshure an equal bias.

In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X, 
    y,
    stratify=y, 
    test_size=0.3,
    random_state=456
)

In [7]:
print(len(X_train), len(X_val))

4890 2097


## 4. Tokenize Dataset
To convert the text into a usable format for the following models, a vectorization is perfomed. The vectorizer is trained on the training-subset and applied onto the validation- and testing-split.

In [8]:
vectorizer = TfidfVectorizer()
X_train_vector = vectorizer.fit_transform(X_train)
X_val_vector = vectorizer.transform(X_val)

## 5. Naive Bayes Classification

### 5.1 Define Naive Bayes Classifier & Parameter Grid
First, the naive bayes classifier model must be defined. In order to a achieve the best classification results, an hyperparameteroptimization is performed. Due to the small optimizationspace, gridsearch is applied. The parameter grid contains all possible hyperparameter combinations.

In [9]:
parameter_grid_nb = {
    'alpha': np.linspace(0.001, 0.1, 100),
    'fit_prior': [True, False]
}

nb_classifier = MultinomialNB()

### 5.2 Grid-Search & Cross-Validation
Next, the hyperparameteroptimization can be performed. In addition, a 5-fold cross validation is used to enshure the best possible generalization of the model.

In [10]:
grid_search_nb = GridSearchCV(nb_classifier, parameter_grid_nb, cv=5)
# grid_search_nb.fit(X_train_vector, y_train)

GridSearchCV(cv=5, estimator=MultinomialNB(),
             param_grid={'alpha': array([0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009,
       0.01 , 0.011, 0.012, 0.013, 0.014, 0.015, 0.016, 0.017, 0.018,
       0.019, 0.02 , 0.021, 0.022, 0.023, 0.024, 0.025, 0.026, 0.027,
       0.028, 0.029, 0.03 , 0.031, 0.032, 0.033, 0.034, 0.035, 0.036,
       0.037, 0.038, 0.039, 0.04 , 0.041, 0.042, 0.043, 0.044, 0.045,
       0.046, 0.047, 0.048, 0.049, 0.05 , 0.051, 0.052, 0.053, 0.054,
       0.055, 0.056, 0.057, 0.058, 0.059, 0.06 , 0.061, 0.062, 0.063,
       0.064, 0.065, 0.066, 0.067, 0.068, 0.069, 0.07 , 0.071, 0.072,
       0.073, 0.074, 0.075, 0.076, 0.077, 0.078, 0.079, 0.08 , 0.081,
       0.082, 0.083, 0.084, 0.085, 0.086, 0.087, 0.088, 0.089, 0.09 ,
       0.091, 0.092, 0.093, 0.094, 0.095, 0.096, 0.097, 0.098, 0.099,
       0.1  ]),
                         'fit_prior': [True, False]})

### 5.3 Validate Classification Results
After the training, the best naive bayes model is extracted along with the hyperparameters and validated via the validation-split of the dataset. The resulting metrics are printed and later compared.

In [18]:
# best_nb = grid_search_nb.best_estimator_
# print("Best Hyperparameter: ", grid_search_nb.best_params_)

# joblib.dump(best_nb, './99_Saved Models/00_naive_bayes_model.joblib') # Save trained classifier
best_nb = joblib.load('./99_Saved Models/00_naive_bayes_model.joblib') # Load trained classifier

Best Hyperparameter:  {'alpha': 0.016, 'fit_prior': True}


['./99_Saved Models/00_naive_bayes_model.joblib']

In [16]:
y_val_predictions_nb = best_nb.predict(X_val_vector)
data = [["F1-Score", "G-Mean", "AUC", "Precision", "Recall", "Accuracy"], calculate_metrics(y_val, y_val_predictions_nb)] # Create list with values
table = PrettyTable(data[0]) # Generate table with metrics
table.add_rows(data[1:]) # Add data to table
print(table) # Show table

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|      F1-Score      |       G-Mean       |        AUC         |     Precision      |       Recall       |      Accuracy      |
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
| 0.8229954614220877 | 0.8479402536208901 | 0.8559275464204301 | 0.7132867132867133 | 0.9725864123957092 | 0.8326180257510729 |
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+


## 6. Support Vector Machine Classification

### 6.1 Define SVM Classifier & Parameter Grid
First, the support vector classification model must be defined. In order to a achieve the best classification results, an hyperparameteroptimization is performed. Due to the small optimizationspace, gridsearch is applied. The parameter grid contains all possible hyperparameter combinations.

In [19]:
parameter_grid_svm = {
    'C': [1, 2, 3, 4, 5],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

svm_classifier = SVC()

### 6.2 Grid-Search & Cross-Validation
Next, the hyperparameteroptimization can be performed. In addition, a 5-fold cross validation is used to enshure the best possible generalization of the model.

In [20]:
grid_search_svm = GridSearchCV(svm_classifier, parameter_grid_svm, cv=5)
# grid_search_svm.fit(X_train_vector, y_train)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [1, 2, 3, 4, 5], 'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf']})

### 6.3 Validate Classification Results
After the training, the best svm classification model is extracted along with the hyperparameters and validated via the validation-split of the dataset. The resulting metrics are printed and later compared.

In [25]:
# best_svm = grid_search_svm.best_estimator_
# print("Best Hyperparameter: ", grid_search_svm.best_params_)

# joblib.dump(best_svm, './99_Saved Models/01_support_vector_model.joblib') # Save trained classifier
best_svm = joblib.load('./99_Saved Models/01_support_vector_model.joblib') # Load trained classifier

Best Hyperparameter:  {'C': 4, 'gamma': 'scale', 'kernel': 'linear'}


['./99_Saved Models/01_support_vector_model.joblib']

In [22]:
y_val_predictions_svm = best_svm.predict(X_val_vector)
data = [["F1-Score", "G-Mean", "AUC", "Precision", "Recall", "Accuracy"], calculate_metrics(y_val, y_val_predictions_svm)] # Create list with values
table = PrettyTable(data[0]) # Generate table with metrics
table.add_rows(data[1:]) # Add data to table
print(table) # Show table

+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+
|      F1-Score      |       G-Mean       |        AUC        |     Precision      |       Recall       |      Accuracy      |
+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+
| 0.9287356321839081 | 0.9443812200064508 | 0.944562191722677 | 0.8967813540510544 | 0.9630512514898689 | 0.9408679065331426 |
+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+


## 7. Decision Tree Classification

### 7.1 Define Decision Tree Classifier & Parameter Grid
First, the decision tree classification model must be defined. In order to a achieve the best classification results, an hyperparameteroptimization is performed. Due to the small optimizationspace, gridsearch is applied. The parameter grid contains all possible hyperparameter combinations.

In [23]:
parameter_grid_dt = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
}

dt_classifier = DecisionTreeClassifier()

### 7.2 Grid-Search & Cross-Validation
Next, the hyperparameteroptimization can be performed. In addition, a 5-fold cross validation is used to enshure the best possible generalization of the model.

In [24]:
grid_search_dt = GridSearchCV(dt_classifier, parameter_grid_dt, cv=5)
grid_search_dt.fit(X_train_vector, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [None, 5, 10, 15],
                         'min_samples_split': [2, 5, 10]})

### 7.3 Validate Classification Results
After the training, the best random forest classification model is extracted along with the hyperparameters and validated via the validation-split of the dataset. The resulting metrics are printed and later compared.

In [26]:
# best_dt = grid_search_dt.best_estimator_
# print("Best Hyperparameter: ", grid_search_dt.best_params_)

# joblib.dump(best_dt, './99_Saved Models/02_decision_tree_model.joblib') # Save trained classifier
best_dt = joblib.load('./99_Saved Models/02_decision_tree_model.joblib') # Load trained classifier

Best Hyperparameter:  {'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 2}


['./99_Saved Models/02_decision_tree_model.joblib']

In [27]:
y_val_predictions_dt = best_dt.predict(X_val_vector)
data = [["F1-Score", "G-Mean", "AUC", "Precision", "Recall", "Accuracy"], calculate_metrics(y_val, y_val_predictions_dt)] # Create list with values
table = PrettyTable(data[0]) # Generate table with metrics
table.add_rows(data[1:]) # Add data to table
print(table) # Show table

+-------------------+--------------------+--------------------+--------------------+-------------------+--------------------+
|      F1-Score     |       G-Mean       |        AUC         |     Precision      |       Recall      |      Accuracy      |
+-------------------+--------------------+--------------------+--------------------+-------------------+--------------------+
| 0.953556731334509 | 0.9630365301012427 | 0.9630431981445092 | 0.9408352668213457 | 0.966626936829559 | 0.9623271340009537 |
+-------------------+--------------------+--------------------+--------------------+-------------------+--------------------+


## 8. Compare Results
In the previous steps, three different classification models were created and optimally trained. A comparison of the results achieved in each case is given in the following table:

In [28]:
data = [
    ["MODEL", "F1-Score", "G-Mean", "AUC", "Precision", "Recall", "Accuracy"],
    ["Naive Bayes"]+calculate_metrics(y_val, y_val_predictions_nb),
    ["Support Vector Machine"]+calculate_metrics(y_val, y_val_predictions_svm),
    ["Decision Tree"]+calculate_metrics(y_val, y_val_predictions_dt)
] # Create list with values

table = PrettyTable(data[0]) # Generate table with metrics
table.add_rows(data[1:]) # Add data to table
print(table) # Show table

+------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|         MODEL          |      F1-Score      |       G-Mean       |        AUC         |     Precision      |       Recall       |      Accuracy      |
+------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|      Naive Bayes       | 0.8229954614220877 | 0.8479402536208901 | 0.8559275464204301 | 0.7132867132867133 | 0.9725864123957092 | 0.8326180257510729 |
| Support Vector Machine | 0.9287356321839081 | 0.9443812200064508 | 0.944562191722677  | 0.8967813540510544 | 0.9630512514898689 | 0.9408679065331426 |
|     Decision Tree      | 0.953556731334509  | 0.9630365301012427 | 0.9630431981445092 | 0.9408352668213457 | 0.966626936829559  | 0.9623271340009537 |
+------------------------+--------------------+--------------------+--------------

It can be seen that the Naive Bayes classification achieves the worst result. Only with the recall metric can the highest values be achieved with this method. In second place is the Support Vector Machine, which can consistently achieve a good classification. However, a Decision Tree Classification performed best and can distinguish Wikipedia articles into excellent and normal articles with an accuracy of about 96%.

## 9. Conclusion
In this notebook, the already pre-processed data was loaded, the data set was split and tokenised. Then a Naive Bayes, a Support Vector Machine and a Decision Tree classification model were created, optimally trained and subsequently validated. The decision tree classifier achieved the best results. These results show that natural language processing is also possible with comparatively simple models.